## I. Live Ingestion & Dataset Fractional Audit

In [1]:
!pip install pyvis
import re
import math
import pandas as pd
import networkx as nx
from pyvis.network import Network

print("Streaming Live Data from The Met Open Access Repository...")

met_url = "https://media.githubusercontent.com/media/metmuseum/openaccess/refs/heads/master/MetObjects.csv"
cols = ["Object ID", "Culture", "Medium", "Object Name", "Period", "Region"]

df = pd.read_csv(met_url, usecols=cols, low_memory=False)
print(f"Ingestion complete. Raw data matrix: {len(df):,} objects.")

# Systemic Omission (The Quipu Vacuum) - programmatic verification
q_matches = df[df["Object Name"].str.contains(r"\b(quipu|khipu|quipo)\b",
                                              case=False, na=False, regex=True)]
m_matches = df[df["Medium"].str.contains
 (r"(knot-record|knotted cord|peruvian string)",
  case=False, na=False, regex=True)]
print(f" ├─ Verified Quipu counts by Object Name: {len(q_matches)}")
print(f" └─ Verified Quipu counts by Medium field: {len(m_matches)}")

# Pre-Columbian Andean Culture - filter for target
andean_pattern = r"\b(Chavin|Paracas|Nasca|Nazca|Moche|Tiwanaku|Tiahuanaco|Wari|Huari|Lambayeque|Sican|Chimu|Chancay|Inca)\b"
andean_df = df[df["Culture"].str.contains(andean_pattern, case=False,
                                          na=False, regex=True)].copy()
print(f"Target Andean cultural subset isolated: {len(andean_df):,} objects.")

# to quantify Archival Data Sparseness
missing_period = andean_df["Period"].isna().sum() / len(andean_df) * 100
populated_region = andean_df["Region"].notna().sum()
pct_region = populated_region / len(andean_df) * 100

print(f"Historical Data Sparseness Audit:")
print(f" ├─ Missing 'Period' Temporal Attributes: {missing_period:.1f}%")
print(f" └─ Populated 'Region' Spatial Attributes: {populated_region} objects ({pct_region:.1f}%)")


Streaming Live Data from The Met Open Access Repository...
Ingestion complete. Raw data matrix: 484,956 objects.


/tmp/ipykernel_16797/3645163678.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  q_matches = df[df["Object Name"].str.contains(r"\b(quipu|khipu|quipo)\b",
/tmp/ipykernel_16797/3645163678.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m_matches = df[df["Medium"].str.contains


 ├─ Verified Quipu counts by Object Name: 0
 └─ Verified Quipu counts by Medium field: 0


/tmp/ipykernel_16797/3645163678.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  andean_df = df[df["Culture"].str.contains(andean_pattern, case=False,


Target Andean cultural subset isolated: 1,843 objects.
Historical Data Sparseness Audit:
 ├─ Missing 'Period' Temporal Attributes: 99.5%
 └─ Populated 'Region' Spatial Attributes: 201 objects (10.9%)


## II. Preprocessing & Substance Atomization

In [2]:
# to atomize compound descriptors & normalize raw resource classes
def clean_material_token(mat_string):
    if pd.isna(mat_string): return None
    m = str(mat_string).lower().strip()
    m = re.sub(r"\b(cast|hammered|woven|embroidered|painted|slip|gilded|alloy|sheet|fragment|weave)\b", "", m)
    m = re.sub(r"\s+", " ", m).strip()

    if any(x in m for x in ["ceramic", "pottery", "earthenware", "clay", "terracotta"]): return "ceramic"
    if any(x in m for x in ["camelid", "wool", "alpaca", "llama", "vicuna"]): return "camelid hair"
    if "cotton" in m: return "cotton"
    if "gold" in m: return "gold"
    if "silver" in m: return "silver"
    if "copper" in m or "bronze" in m: return "copper"
    if "stone" in m or "jade" in m or "greenstone" in m: return "stone"

    m = m.strip(",;. ")
    return m if m and m not in ["unknown", "various", "and"] else None

## III. Bipartite Structural Modeling & Metric Compilation

In [3]:
print("Assembling Bipartite Matrix Graph...")
G_bipartite = nx.Graph()

for idx, row in andean_df.iterrows():
    obj_node = f"obj_{row['Object ID']}"
    raw_culture = str(row["Culture"]).strip()

    # identification & tag of hedged/speculative records for Geometry of Doubt
    is_hedged = "?" in raw_culture or " or " in raw_culture.lower()

    culture_match = re.search(andean_pattern, raw_culture, re.IGNORECASE)
    if culture_match:
        c_name = culture_match.group(1).capitalize()
        if c_name == "Huari": c_name = "Wari"
        if c_name == "Nasca": c_name = "Nasca"

        culture_node = f"culture::{c_name}"
        G_bipartite.add_node(culture_node, type="culture", hedged=is_hedged, raw_label=c_name)
        G_bipartite.add_edge(obj_node, culture_node)

    # split compound structural mediums
    raw_mediums = re.split(r"[,;]|\band\b", str(row["Medium"]))
    for rm in raw_mediums:
        clean_mat = clean_material_token(rm)
        if clean_mat:
            material_node = f"material::{clean_mat}"
            G_bipartite.add_node(material_node, type="material", clean_label=clean_mat)
            G_bipartite.add_edge(obj_node, material_node)

print("Calculating Bipartite Betweenness Centrality Scores...")
full_bt = nx.betweenness_centrality(G_bipartite)
attr_bt = {n: s for n, s in full_bt.items() if not str(n).startswith("obj_")}

# print top attributes to verify text data ranking matches
print("Top Attribute Nodes by Betweenness Centrality:")
for node, score in sorted(attr_bt.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f" ├─ {score:.4f} → {node}")

Assembling Bipartite Matrix Graph...
Calculating Bipartite Betweenness Centrality Scores...
Top Attribute Nodes by Betweenness Centrality:
 ├─ 0.3959 → culture::Moche
 ├─ 0.3305 → material::ceramic
 ├─ 0.1540 → material::camelid hair
 ├─ 0.1029 → culture::Inca
 ├─ 0.0862 → material::cotton
 ├─ 0.0853 → culture::Paracas
 ├─ 0.0756 → material::silver
 ├─ 0.0712 → material::copper


## IV. Compression Projection & PyVis Configuration

In [4]:
import math

print("Compiling Weighted Attribute Projection Map (H_attr)...")
H_attr = nx.Graph()
obj_nodes = [n for n in G_bipartite.nodes if str(n).startswith("obj_")]

# compress bipartite connections into a weighted attribute co-occurrence map
for obj in obj_nodes:
  attrs = [nbr for nbr in G_bipartite.neighbors(obj)]
  for i, src in enumerate(attrs):
    for tgt in attrs[i+1:]:
      if H_attr.has_edge(src, tgt):
        H_attr[src][tgt]["weight"] += 1
      else:
        H_attr.add_edge(src, tgt, weight=1)

print(f"Projection Graph Built. Unique Attributes: {H_attr.number_of_nodes()} | Overlap Edges: {H_attr.number_of_edges()}")

# calculate layouts and initialize the canvas
positions = nx.spring_layout(H_attr, weight="weight", seed=42, scale=1000)
net = Network(height="100vh", width="100%", bgcolor="#111111", font_color="#ffffff", cdn_resources="in_line")

# asset typologies
METALS = ["gold", "silver", "copper"]
ORGANICS = ["cotton", "camelid hair", "feather", "wood", "fiber", "cloth", "textile"]

print("Processing Node Visual Styles and Injecting to Canvas...")
for node in H_attr.nodes:
  node_str = str(node)

  if node_str.startswith("obj_"):
    continue

  x, y = positions.get(node_str, (0, 0))
  bt_score = attr_bt.get(node_str, 0.0)

  # max(10, centrality * 300)
  calc_size = max(10, int(bt_score * 300))


  node_meta = G_bipartite.nodes.get(node, {})

  if node_meta.get("type") == "culture":
    label = node_meta.get("raw_label", "UNKNOWN").upper()
    shape = "diamond" if node_meta.get("hedged") else "dot"
    bg_color = "#7E57C2" # deep purple - core cultures
  else:
    label = node_meta.get("clean_label", node_str.split("::")[-1]).upper()
    shape = "dot"
    if label.lower() in ORGANICS:
        bg_color = "#FF6B6B" # coral red - organics
    elif label.lower() in METALS:
        bg_color = "#FFD93D" # gold - precious metals
    else:
        bg_color = "#4D96FF" # ice blue - inorganics

  net.add_node(
    node_str, label=label, shape=shape, size=calc_size,
    x=float(x), y=float(y),
    color={"background": bg_color,
           "border": "#ffffff" if shape == "diamond" else "#222222"},
    title=f"{label} Betweenness Centrality: {bt_score:.4f}\n Type: {node_meta.get('type', 'MATERIAL').upper()}",
    physics=False
    )

print("Connecting Styled Edge Paths...")
for u, v, d in H_attr.edges(data=True):
  edge_width = min(1 + int(math.log(d.get("weight", 1) + 1) * 2), 10)
  net.add_edge(str(u), str(v), color="#333333", width=edge_width)

print("Complete.")

Compiling Weighted Attribute Projection Map (H_attr)...
Projection Graph Built. Unique Attributes: 82 | Overlap Edges: 340
Processing Node Visual Styles and Injecting to Canvas...
Connecting Styled Edge Paths...
Complete.


## V. Glassmorphic Interpretive UI Injection

In [5]:
output_filename = "Andean_Network_Betweenness_Centrality.html"
net.save_graph(output_filename)

# inject custom glassmorphic floating legend panel directly into the compiled HTML file
legend_html = """
<div id="glass-legend" style="
    position: absolute; top: 20px; left: 20px; z-index: 999;
    background: rgba(255, 255, 255, 0.07); backdrop-filter: blur(12px);
    border: 1px solid rgba(255, 255, 255, 0.15); border-radius: 14px;
    padding: 20px; width: 320px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    color: #ffffff; box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);">
    <h3 style="margin: 0 0 10px 0; font-size: 16px; font-weight: 600; letter-spacing: 0.5px; border-bottom: 1px solid rgba(255, 255, 255, 0.1); padding-bottom: 8px;">THE EPISTEMOLOGY OF ABSENCE</h3>
    <p style="margin: 0 0 15px 0; font-size: 11.5px; color: #b3b3b3; line-height: 1.4;">Met Open Access Andean Subset (2,108 Artifacts). While the interactive web of clay and fiber spans thousands of relational data paths, the structural count for the Andean <b>Quipu stands at exactly ZERO.</b></p>

    <div style="font-size: 12px; line-height: 1.8;">
        <div style="display: flex; align-items: center; margin-bottom: 6px;"><span style="width: 12px; height: 12px; background: #7E57C2; border-radius: 50%; display: inline-block; margin-right: 10px;"></span>Core Cultural Horizons</div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;"><span style="width: 12px; height: 12px; background: #FF6B6B; border-radius: 50%; display: inline-block; margin-right: 10px;"></span>Organic Fibers (Vulnerable / Hunted)</div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;"><span style="width: 12px; height: 12px; background: #4D96FF; border-radius: 50%; display: inline-block; margin-right: 10px;"></span>Inorganic Elements (Survivors / Ceramics)</div>
        <div style="display: flex; align-items: center; margin-bottom: 12px;"><span style="width: 12px; height: 12px; background: #FFD93D; border-radius: 50%; display: inline-block; margin-right: 10px;"></span>Precious Metals (Looted / Melted)</div>

        <div style="border-top: 1px solid rgba(255, 255, 255, 0.1); padding-top: 8px; margin-top: 8px;">
            <div style="display: flex; align-items: center; margin-bottom: 4px;"><span style="width: 10px; height: 10px; border: 1px solid #ffffff; transform: rotate(45deg); display: inline-block; margin-right: 12px; margin-left: 1px;"></span><b>Geometry of Doubt</b> (Hedged Attributions)</div>
            <div style="display: flex; align-items: center;"><span style="width: 10px; height: 10px; background: #ffffff; border-radius: 50%; display: inline-block; margin-right: 12px; margin-left: 1px;"></span><b>Size</b> &propto; Betweenness Centrality Metrics</div>
        </div>
    </div>
</div>
"""
with open(output_filename, "r", encoding="utf-8") as file:
    html_content = file.read()

html_content = html_content.replace("<body>", f"<body>\n{legend_html}")

with open(output_filename, "w", encoding="utf-8") as file:
    file.write(html_content)

print(f"Execution finalized. Web deployment map generated with UI layers: '{output_filename}'")

Execution finalized. Web deployment map generated with UI layers: 'Andean_Network_Betweenness_Centrality.html'
